# Knapsack Problem in Pure Python
## Five Exact Methods for Book Problem 2.12


## Problem Statement

Book problem `2.12` is a binary knapsack problem: a thief chooses which products to steal so that the
total volume does not exceed `1.1 m^3` and the resale profit is maximized.

The student variation adds pair constraints: phones must travel together with speakers, laptops with
the sound system, and the microwave with the television.


In [ ]:
from __future__ import annotations

from functools import lru_cache, wraps
from time import perf_counter


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def choose_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (left["total_profit"], -left["total_volume"], tuple(left["selected_original_items"]))
    right_key = (right["total_profit"], -right["total_volume"], tuple(right["selected_original_items"]))
    return right if right_key > left_key else left


BASE_ITEMS = [
    {"key": "phone_box", "label": "Box of phones", "volume": 17, "profit": 2_000_000, "members": ["phone_box"]},
    {"key": "laptop_dozen", "label": "Dozen laptops", "volume": 50, "profit": 6_000_000, "members": ["laptop_dozen"]},
    {"key": "book_box", "label": "Box of books", "volume": 32, "profit": 900_000, "members": ["book_box"]},
    {"key": "cream_box", "label": "Box of creams", "volume": 9, "profit": 1_100_000, "members": ["cream_box"]},
    {"key": "sound_system", "label": "Sound system", "volume": 20, "profit": 900_000, "members": ["sound_system"]},
    {"key": "car_battery", "label": "Car battery", "volume": 25, "profit": 1_600_000, "members": ["car_battery"]},
    {"key": "microwave", "label": "Microwave", "volume": 1, "profit": 300_000, "members": ["microwave"]},
    {"key": "television", "label": "Television", "volume": 40, "profit": 2_500_000, "members": ["television"]},
    {"key": "speakers", "label": "Speakers", "volume": 28, "profit": 1_300_000, "members": ["speakers"]},
]

VARIANT_ITEMS = [
    {"key": "phones_plus_speakers", "label": "Phones + speakers", "volume": 45, "profit": 3_300_000, "members": ["phone_box", "speakers"]},
    {"key": "laptops_plus_sound", "label": "Laptops + sound system", "volume": 70, "profit": 6_900_000, "members": ["laptop_dozen", "sound_system"]},
    {"key": "microwave_plus_television", "label": "Microwave + television", "volume": 41, "profit": 2_800_000, "members": ["microwave", "television"]},
    {"key": "book_box", "label": "Box of books", "volume": 32, "profit": 900_000, "members": ["book_box"]},
    {"key": "cream_box", "label": "Box of creams", "volume": 9, "profit": 1_100_000, "members": ["cream_box"]},
    {"key": "car_battery", "label": "Car battery", "volume": 25, "profit": 1_600_000, "members": ["car_battery"]},
]

BASE_PROBLEM = {"capacity": 110, "items": BASE_ITEMS}
VARIANT_PROBLEM = {"capacity": 110, "items": VARIANT_ITEMS}

EXPECTED_BASE = {
    "selected_grouped_items": ["car_battery", "cream_box", "laptop_dozen", "microwave", "phone_box"],
    "selected_original_items": ["car_battery", "cream_box", "laptop_dozen", "microwave", "phone_box"],
    "total_profit": 11_000_000,
    "total_volume": 102,
}
EXPECTED_VARIANT = {
    "selected_grouped_items": ["car_battery", "cream_box", "laptops_plus_sound"],
    "selected_original_items": ["car_battery", "cream_box", "laptop_dozen", "sound_system"],
    "total_profit": 9_600_000,
    "total_volume": 104,
}


def build_solution(problem, selected_indexes):
    items = problem["items"]
    selected = [items[index] for index in selected_indexes]
    total_volume = sum(item["volume"] for item in selected)
    total_profit = sum(item["profit"] for item in selected)
    if total_volume > problem["capacity"]:
        return None
    original = sorted(member for item in selected for member in item["members"])
    grouped = sorted(item["key"] for item in selected)
    return {
        "selected_grouped_items": grouped,
        "selected_original_items": original,
        "total_profit": total_profit,
        "total_volume": total_volume,
    }


def density_sorted(problem):
    return sorted(problem["items"], key=lambda item: item["profit"] / item["volume"], reverse=True)


@timed
def solve_knapsack_naive(problem):
    best = None
    item_count = len(problem["items"])
    for mask in range(1 << item_count):
        selected = [index for index in range(item_count) if (mask >> index) & 1]
        candidate = build_solution(problem, selected)
        best = choose_better(best, candidate)
    return best


@timed
def solve_knapsack_backtracking(problem):
    ordered_items = density_sorted(problem)
    best = None

    def optimistic_profit(start, capacity_left):
        total = 0.0
        for item in ordered_items[start:]:
            if item["volume"] <= capacity_left:
                total += item["profit"]
                capacity_left -= item["volume"]
            else:
                total += item["profit"] * capacity_left / item["volume"]
                break
        return total

    def backtrack(index, selected_keys, used_volume, current_profit):
        nonlocal best
        capacity_left = problem["capacity"] - used_volume
        bound = current_profit + optimistic_profit(index, capacity_left)
        if best is not None and bound <= best["total_profit"]:
            return
        if index == len(ordered_items):
            selected_indexes = [next(i for i, item in enumerate(problem["items"]) if item["key"] == key) for key in selected_keys]
            candidate = build_solution(problem, selected_indexes)
            best = choose_better(best, candidate)
            return
        item = ordered_items[index]
        backtrack(index + 1, selected_keys, used_volume, current_profit)
        if used_volume + item["volume"] <= problem["capacity"]:
            backtrack(index + 1, selected_keys + [item["key"]], used_volume + item["volume"], current_profit + item["profit"])

    backtrack(0, [], 0, 0)
    return best


@timed
def solve_knapsack_reduced_base():
    reduced_problem = {
        "capacity": BASE_PROBLEM["capacity"] - 50,
        "items": [item for item in BASE_ITEMS if item["key"] != "laptop_dozen"],
    }
    reduced_solution = solve_knapsack_naive(reduced_problem)[0]
    solution = {
        "selected_grouped_items": sorted(["laptop_dozen"] + reduced_solution["selected_grouped_items"]),
        "selected_original_items": sorted(["laptop_dozen"] + reduced_solution["selected_original_items"]),
        "total_profit": 6_000_000 + reduced_solution["total_profit"],
        "total_volume": 50 + reduced_solution["total_volume"],
    }
    return solution


@timed
def solve_knapsack_reduced_variant():
    best = None
    item_count = len(VARIANT_ITEMS)
    for mask in range(1 << item_count):
        selected = [index for index in range(item_count) if (mask >> index) & 1]
        candidate = build_solution(VARIANT_PROBLEM, selected)
        best = choose_better(best, candidate)
    return best


@timed
def solve_knapsack_dynamic_programming(problem):
    items = problem["items"]
    capacity = problem["capacity"]
    dp = [[None] * (capacity + 1) for _ in range(len(items) + 1)]
    dp[0][0] = []
    for index, item in enumerate(items, start=1):
        for volume in range(capacity + 1):
            if dp[index - 1][volume] is not None:
                current = list(dp[index - 1][volume])
                if dp[index][volume] is None:
                    dp[index][volume] = current
            if volume >= item["volume"] and dp[index - 1][volume - item["volume"]] is not None:
                candidate = dp[index - 1][volume - item["volume"]] + [index - 1]
                existing = build_solution(problem, dp[index][volume]) if dp[index][volume] is not None else None
                candidate_solution = build_solution(problem, candidate)
                if existing is None or choose_better(existing, candidate_solution) is candidate_solution:
                    dp[index][volume] = candidate
    best = None
    for volume in range(capacity + 1):
        if dp[len(items)][volume] is not None:
            best = choose_better(best, build_solution(problem, dp[len(items)][volume]))
    return best


@timed
def solve_knapsack_branch_and_bound(problem):
    ordered_items = density_sorted(problem)
    best = solve_knapsack_backtracking(problem)[0]

    def optimistic_profit(start, capacity_left):
        total = 0.0
        for item in ordered_items[start:]:
            if item["volume"] <= capacity_left:
                total += item["profit"]
                capacity_left -= item["volume"]
            else:
                total += item["profit"] * capacity_left / item["volume"]
                break
        return total

    def branch(index, selected_keys, used_volume, current_profit):
        nonlocal best
        capacity_left = problem["capacity"] - used_volume
        bound = current_profit + optimistic_profit(index, capacity_left)
        if bound <= best["total_profit"]:
            return
        if index == len(ordered_items):
            selected_indexes = [next(i for i, item in enumerate(problem["items"]) if item["key"] == key) for key in selected_keys]
            candidate = build_solution(problem, selected_indexes)
            best = choose_better(best, candidate)
            return
        item = ordered_items[index]
        if used_volume + item["volume"] <= problem["capacity"]:
            branch(index + 1, selected_keys + [item["key"]], used_volume + item["volume"], current_profit + item["profit"])
        branch(index + 1, selected_keys, used_volume, current_profit)

    branch(0, [], 0, 0)
    return best


## 1. Naive Enumeration


In [ ]:
base_naive_result, base_naive_time = solve_knapsack_naive(BASE_PROBLEM)
variant_naive_result, variant_naive_time = solve_knapsack_naive(VARIANT_PROBLEM)
print("Base result:", base_naive_result)
print("Variation result:", variant_naive_result)
print(f"Times -> base={base_naive_time:.6f}s variation={variant_naive_time:.6f}s")
assert base_naive_result == EXPECTED_BASE
assert variant_naive_result == EXPECTED_VARIANT


## 2. Backtracking With Pruning


In [ ]:
base_backtracking_result, base_backtracking_time = solve_knapsack_backtracking(BASE_PROBLEM)
variant_backtracking_result, variant_backtracking_time = solve_knapsack_backtracking(VARIANT_PROBLEM)
print("Base result:", base_backtracking_result)
print("Variation result:", variant_backtracking_result)
print(f"Times -> base={base_backtracking_time:.6f}s variation={variant_backtracking_time:.6f}s")
assert base_backtracking_result == EXPECTED_BASE
assert variant_backtracking_result == EXPECTED_VARIANT


## 3. Constraint-Driven Reduced Search


In [ ]:
best_without_laptops = solve_knapsack_naive({"capacity": 110, "items": [item for item in BASE_ITEMS if item["key"] != "laptop_dozen"]})[0]
print("Best base solution without laptops:", best_without_laptops)
assert best_without_laptops["total_profit"] < EXPECTED_BASE["total_profit"]

base_reduced_result, base_reduced_time = solve_knapsack_reduced_base()
variant_reduced_result, variant_reduced_time = solve_knapsack_reduced_variant()
print("Base reduced-search result:", base_reduced_result)
print("Variation reduced-search result:", variant_reduced_result)
print(f"Times -> base={base_reduced_time:.6f}s variation={variant_reduced_time:.6f}s")
assert base_reduced_result == EXPECTED_BASE
assert variant_reduced_result == EXPECTED_VARIANT


## 4. Dynamic Programming


In [ ]:
base_dp_result, base_dp_time = solve_knapsack_dynamic_programming(BASE_PROBLEM)
variant_dp_result, variant_dp_time = solve_knapsack_dynamic_programming(VARIANT_PROBLEM)
print("Base result:", base_dp_result)
print("Variation result:", variant_dp_result)
print(f"Times -> base={base_dp_time:.6f}s variation={variant_dp_time:.6f}s")
assert base_dp_result == EXPECTED_BASE
assert variant_dp_result == EXPECTED_VARIANT


## 5. Branch And Bound


In [ ]:
base_bnb_result, base_bnb_time = solve_knapsack_branch_and_bound(BASE_PROBLEM)
variant_bnb_result, variant_bnb_time = solve_knapsack_branch_and_bound(VARIANT_PROBLEM)
print("Base result:", base_bnb_result)
print("Variation result:", variant_bnb_result)
print(f"Times -> base={base_bnb_time:.6f}s variation={variant_bnb_time:.6f}s")
assert base_bnb_result == EXPECTED_BASE
assert variant_bnb_result == EXPECTED_VARIANT


## Summary


In [ ]:
table = [
    ("Naive enumeration", base_naive_result, base_naive_time, variant_naive_result, variant_naive_time),
    ("Backtracking with pruning", base_backtracking_result, base_backtracking_time, variant_backtracking_result, variant_backtracking_time),
    ("Constraint-driven reduced search", base_reduced_result, base_reduced_time, variant_reduced_result, variant_reduced_time),
    ("Dynamic programming", base_dp_result, base_dp_time, variant_dp_result, variant_dp_time),
    ("Branch and Bound", base_bnb_result, base_bnb_time, variant_bnb_result, variant_bnb_time),
]

for method_name, base_result, base_time, variant_result, variant_time in table:
    print(
        f"{method_name:35s} "
        f"base_profit={base_result['total_profit']:8d} base_time={base_time:.6f}s "
        f"variation_profit={variant_result['total_profit']:8d} variation_time={variant_time:.6f}s"
    )

assert all(base_result == EXPECTED_BASE for _, base_result, _, _, _ in table)
assert all(variant_result == EXPECTED_VARIANT for _, _, _, variant_result, _ in table)
print("\nThe variation is solved exactly after reformulating the mandatory pairs as composite items.")
